# Task 4 · Point-to-surface evaluation
Use a CPU runtime with the Drive folder from the Task 3 generator and a full
Task 4 training run. The notebook scores all 12 test objects against their
original meshes and then scores the marked points on the two texture-only
objects. Set TASK4_RUN to select an exact run for a paired comparison.


In [ ]:
from pathlib import Path
from google.colab import drive
import json
import os
import subprocess
import sys

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/MVP_Project')
TASK4_RUN = ''  # Set to a full_* run folder name. The default uses the latest complete run.
data = PROJECT / 'task3/gso_task3_v1'
runs = PROJECT / 'task4'

def complete(path):
    required = ('run.json', 'per_object.csv', 'geometry.pt', 'plain_rgb.pt', 'gated_rgb.pt')
    return all((path / filename).is_file() for filename in required)

if TASK4_RUN:
    run = runs / TASK4_RUN
    if not complete(run):
        raise FileNotFoundError(f'Incomplete Task 4 run: {run}')
else:
    full_runs = sorted((p for p in runs.glob('full_*') if complete(p)), reverse=True)
    if not full_runs:
        raise FileNotFoundError(f'Run the Task 4 training notebook first. No complete runs in {runs}')
    run = full_runs[0]

manifest = json.loads((data / 'manifest.json').read_text())
candidates = data.parent / f"candidates_{manifest['revision'][:8]}_{manifest['points']}_{manifest['seed']}_v1"
pgd_cache = runs / 'pgd_cache'
for required in (candidates, pgd_cache):
    if not required.is_dir():
        raise FileNotFoundError(required)
print('Evaluating:', run)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trimesh==5.2.2', 'point-cloud-utils'], check=True)


In [ ]:
sources = {'data/ranking.py': 'import numpy as np\nimport pandas as pd\nfrom sklearn.neighbors import NearestNeighbors\n\nDATASET_COMPOSITION = (20, 10, 10, 20) # agreement, disagreement (geometry varies), disagreement (texture varies), random samples\n\ndef compute_scores(xyz, rgb, k=24):\n    # Find neighbours in clean 3D geometry.\n    knn = NearestNeighbors(n_neighbors=k + 1, algorithm="kd_tree").fit(xyz)\n    indices = knn.kneighbors(xyz, return_distance=False)[:, 1:]\n    neighbours = xyz[indices]  # (N, k, 3)\n\n    # Local geometric variation via PCA.\n    centered = neighbours - neighbours.mean(axis=1, keepdims=True)\n    covariance = (np.einsum("nki,nkj->nij", centered, centered) / k)\n    eigenvalues = np.linalg.eigvalsh(covariance)\n    geometry_score = (eigenvalues[:, 0] / (eigenvalues.sum(axis=1) + 1e-12))\n\n    # Local RGB variation.\n    color_distances = np.linalg.norm(rgb[indices] - rgb[:, None, :], axis=2)\n    color_score = color_distances.mean(axis=1)\n\n    return geometry_score, color_score\n\ndef compute_thresholds(objects):\n    """\n    objects: dict mapping object_id -> (geometry_scores, colour_scores)\n    """\n    all_g = np.concatenate([g for g, c in objects.values()])\n    all_c = np.concatenate([c for g, c in objects.values()])\n\n    return {\n        "g_high": np.quantile(all_g, 0.90),\n        "g_low": np.quantile(all_g, 0.50),\n        "c_high": np.quantile(all_c, 0.90),\n        "c_low": np.quantile(all_c, 0.50),\n    }\n\n\ndef summarize_object(g, c, thresholds):\n    gh = thresholds["g_high"]\n    gl = thresholds["g_low"]\n    ch = thresholds["c_high"]\n    cl = thresholds["c_low"]\n\n    texture_only = np.mean((c >= ch) & (g <= gl))\n    geometry_only = np.mean((g >= gh) & (c <= cl))\n    aligned = np.mean((g >= gh) & (c >= ch))\n\n    return {\n        "texture_only": float(texture_only),\n        "geometry_only": float(geometry_only),\n        "aligned": float(aligned),\n        "disagreement": float(texture_only + geometry_only),\n    }\n\ndef select_objects(ranking, seed=2025, composition=DATASET_COMPOSITION):\n    selected = set()\n    groups = {}\n\n    agreement, geometry_only, texture_only, random_samples = composition\n\n    criteria = [\n        ("agreement", "aligned", agreement),\n        ("texture_only", "texture_only", texture_only),\n        ("geometry_only", "geometry_only", geometry_only),\n    ]\n\n    for group, column, count in criteria:\n        ranked = ranking.sort_values([column, "object_id"], ascending=[False, True])\n        choices = [obj for obj in ranked["object_id"] if obj not in selected][:count]\n\n        if len(choices) != count:\n            raise ValueError(f"Not enough objects for {group}")\n\n        groups[group] = choices\n        selected.update(choices)\n\n    remaining = ranking.loc[~ranking["object_id"].isin(selected), "object_id", ].to_numpy()\n    rng = np.random.default_rng(seed)\n    groups["random"] = list(rng.choice(remaining, size=random_samples, replace=False))\n\n    return groups\n\ndef create_splits(groups, seed=2025):\n    rng = np.random.default_rng(seed)\n    assignments = {}\n\n    for group, ids in groups.items():\n        ids = list(ids)\n        rng.shuffle(ids)\n\n        n_train = int(len(ids) * 0.7)\n        n_val = int(len(ids) * 0.1)\n\n        for i, object_id in enumerate(ids):\n            split = (\n                "train" if i < n_train\n                else "val" if i < n_train + n_val\n                else "test"\n            )\n\n            assignments[object_id] = {"split": split, "group": group}\n\n    return assignments\n\ndef rank_candidates(candidates):\n    objects = {}\n\n    for candidate in candidates:\n        points, colors, meta = candidate\n        object_id = str(meta["object_id"])\n        objects[object_id] = compute_scores(points, colors)\n\n    thresholds = compute_thresholds(objects)\n\n    rows = [\n            {"object_id": object_id,\n             **summarize_object(g, c, thresholds)}\n            for object_id, (g, c) in objects.items()\n        ]\n    \n    ranking = pd.DataFrame(rows)\n    return ranking', 'task4/train_rgb_pgd.py': '"""Train matched geometry, RGB and gated RGB residual heads on frozen PGD."""\nimport argparse\nimport csv\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nMODES = ("geometry", "plain_rgb", "gated_rgb")\nXYZ_LEVELS = (0.01, 0.02, 0.025)\nRGB_LEVELS = (0.0, 0.05)\nPATCH_SIZE = 1000\nPATCH_OVERSAMPLING = 6\nNEIGHBORS = 16\n\n\ndef mlp(inputs, outputs):\n    return nn.Sequential(nn.Linear(inputs, 64), nn.ReLU(),\n                         nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, outputs))\n\n\nclass ResidualHead(nn.Module):\n    """Same trainable layers in all modes. Only the inputs and gate pooling differ."""\n\n    def __init__(self, mode):\n        super().__init__()\n        if mode not in MODES:\n            raise ValueError(f"Unknown model: {mode}")\n        self.mode = mode\n        self.geometry = mlp(9, 3)\n        self.fusion = mlp(15, 4)\n        for head in (self.geometry, self.fusion):\n            nn.init.zeros_(head[-1].weight)\n            nn.init.zeros_(head[-1].bias)\n\n    def forward(self, base, noisy, indices, rgb):\n        relative = base[indices] - base[:, None]\n        local_geometry = torch.cat(\n            [relative.mean(dim=1), relative.abs().mean(dim=1)], dim=-1\n        )\n        geometry = torch.cat([base - noisy, local_geometry], dim=-1)\n        if self.mode == "geometry":\n            context = local_geometry\n        else:\n            relative_rgb = rgb[indices] - rgb[:, None]\n            context = torch.cat(\n                [relative_rgb.mean(dim=1), relative_rgb.abs().mean(dim=1)], dim=-1\n            )\n\n        mixed = self.fusion(torch.cat([geometry, context], dim=-1))\n        reference = self.fusion(torch.cat([geometry, torch.zeros_like(context)], dim=-1))\n        # Isolate the change caused by context. Uniform RGB gives zero colour correction.\n        contextual_delta = mixed[:, :3] - reference[:, :3]\n        logits = mixed[:, 3:]\n        if self.mode != "gated_rgb":\n            logits = logits.mean(dim=0, keepdim=True).expand_as(logits)\n        gain = logits.sigmoid()\n        return base + self.geometry(geometry) + gain * contextual_delta, gain\n\n\n@dataclass\nclass Sample:\n    oid: str\n    split: str\n    group: str\n    noise: float\n    xyz: np.ndarray\n    clean: np.ndarray\n    colors: dict\n    metric_scale: float\n\n\ndef load_cloud(path):\n    with np.load(path, allow_pickle=False) as arrays:\n        xyz, rgb = (arrays[key].astype(np.float32) for key in ("xyz", "rgb"))\n    if (xyz.shape != rgb.shape or xyz.ndim != 2 or xyz.shape[1] != 3 or\n            not np.isfinite(xyz).all() or not np.isfinite(rgb).all() or\n            np.any(rgb < 0) or np.any(rgb > 1)):\n        raise ValueError(f"Invalid XYZRGB arrays in {path}")\n    return xyz, rgb\n\n\ndef load_samples(root, smoke):\n    assignments = json.loads((root / "selection.json").read_text())\n    split_ids = {split: sorted(oid for oid, info in assignments.items() if info["split"] == split)\n                 for split in ("train", "val", "test")}\n    if set().union(*map(set, split_ids.values())) != set(assignments):\n        raise ValueError("Unexpected split label in selection.json")\n    samples = {}\n    for split, names in split_ids.items():\n        minimum = 2 if split == "train" else 1\n        if len(names) < minimum:\n            raise ValueError(f"{split} requires at least {minimum} distinct objects")\n        samples[split] = []\n        for oid in names[:minimum] if smoke else names:\n            if Path(oid).name != oid or oid in {".", ".."}:\n                raise ValueError(f"Invalid object ID: {oid}")\n            folder = root / split / oid\n            clean, _ = load_cloud(folder / "clean.npz")\n            clean_center = (clean.max(0) + clean.min(0)) / 2\n            clean_radius = np.linalg.norm(clean - clean_center, axis=1).max()\n            if len(clean) < PATCH_SIZE or clean_radius <= 0:\n                raise ValueError(f"Invalid clean point cloud for {oid}")\n            for noise in (0.02,) if smoke else XYZ_LEVELS:\n                xyz, original = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                other_xyz, jitter = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n                if xyz.shape != clean.shape or not np.array_equal(xyz, other_xyz):\n                    raise ValueError(f"Point count or paired geometry mismatch for {oid} at {noise}")\n                center = (xyz.max(0) + xyz.min(0)) / 2\n                radius = np.linalg.norm(xyz - center, axis=1).max()\n                if radius <= 0:\n                    raise ValueError(f"Degenerate input cloud for {oid}")\n                samples[split].append(Sample(\n                    oid, split, assignments[oid]["group"], noise,\n                    (xyz - center) / radius, (clean - center) / radius,\n                    {"original": original, "jitter": jitter}, float(radius / clean_radius),\n                ))\n    return samples\n\n\ndef pgd_patches(model, points, knn, fps, iterations):\n    """Stitch predictions at their original point indices so each RGB stays aligned."""\n    center_count = max(1, PATCH_OVERSAMPLING * len(points) // PATCH_SIZE)\n    best = points.new_full((len(points),), float("inf"))\n    for _ in range(iterations):\n        centers, _ = fps(points[None], center_count)\n        result = torch.empty_like(points)\n        best.fill_(float("inf"))\n\n        def apply_patches(patch_centers):\n            distances, indices, neighborhoods = knn(\n                patch_centers[None], points[None], K=PATCH_SIZE, return_nn=True,\n            )\n            for center, dists, ids, patch in zip(\n                patch_centers, distances[0], indices[0], neighborhoods[0]\n            ):\n                relative = dists / dists[-1].clamp_min(1e-12)\n                predicted = model.denoise_langevin_dynamics((patch - center)[None])[0] + center\n                take = relative < best[ids]\n                result[ids[take]] = predicted[take]\n                best[ids[take]] = relative[take]\n\n        apply_patches(centers[0])\n        # FPS patches can miss a few rows. Process centred patches for any such rows.\n        for idx in torch.where(~best.isfinite())[0].tolist():\n            if not torch.isfinite(best[idx]):\n                apply_patches(points[idx:idx + 1])\n        if not best.isfinite().all() or not result.isfinite().all():\n            raise RuntimeError("PGD stitching failed to reconstruct every input row")\n        points = result\n    return points\n\n\ndef digest_file(path):\n    with path.open("rb") as stream:\n        sha = hashlib.sha256()\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            sha.update(chunk)\n    return sha.hexdigest()\n\n\ndef precompute(samples, pgd_root, checkpoint, iterations, cache, seed, device, knn):\n    sys.path.insert(0, str(pgd_root))\n    from models.pgd import PGDModel\n    from models.utils import farthest_point_sampling\n\n    revision = subprocess.check_output(["git", "-C", str(pgd_root), "rev-parse", "HEAD"], text=True).strip()\n    pgd_config = dict(revision=revision, weights=digest_file(checkpoint), iterations=iterations,\n                      patch_size=PATCH_SIZE, oversampling=PATCH_OVERSAMPLING, seed=seed,\n                      stitch="indexed_v1", device=torch.cuda.get_device_name(device),\n                      torch=torch.__version__)\n    model = None\n    cache.mkdir(parents=True, exist_ok=True)\n    prepared = {}\n    started = time.monotonic()\n    all_samples = [sample for split in samples.values() for sample in split]\n    for number, sample in enumerate(all_samples, 1):\n        key = hashlib.sha256(json.dumps(pgd_config, sort_keys=True).encode() + sample.xyz.tobytes())\n        filename = cache / f"{key.hexdigest()}.npy"\n        if filename.is_file():\n            baseline = np.load(filename, allow_pickle=False)\n        else:\n            if model is None:\n                model = PGDModel.load_from_checkpoint(str(checkpoint), map_location="cpu")\n                model = model.to(device).requires_grad_(False).eval()\n            # Stable result for each cache entry independently of processing order.\n            torch.manual_seed(seed)\n            with torch.no_grad():\n                baseline = pgd_patches(model, torch.from_numpy(sample.xyz).to(device), knn,\n                                       farthest_point_sampling, iterations).cpu().numpy()\n            partial = filename.with_suffix(".part")\n            with partial.open("wb") as stream:\n                np.save(stream, baseline)\n            partial.replace(filename)\n        if baseline.shape != sample.xyz.shape or not np.isfinite(baseline).all():\n            raise ValueError(f"Invalid frozen prediction for {sample.oid}")\n        xyz = torch.from_numpy(sample.xyz.copy()).to(device)\n        with torch.no_grad():\n            ids = knn(xyz[None], xyz[None], K=NEIGHBORS + 1).idx[0]\n            self_index = torch.arange(len(xyz), device=device)[:, None]\n            order = (ids == self_index).to(torch.int32).argsort(dim=1, stable=True)\n            neighbors = ids.gather(1, order[:, :NEIGHBORS]).cpu()\n        prepared[(sample.oid, sample.noise)] = (torch.from_numpy(baseline), neighbors)\n        print(f"PGD {number}/{len(all_samples)} · {sample.split}/{sample.oid} · {sample.noise:g}", flush=True)\n    print(f"PGD preparation took {time.monotonic() - started:.1f} s", flush=True)\n    return prepared, pgd_config\n\n\ndef batch(sample, prepared, condition, device):\n    base, neighbors = prepared[(sample.oid, sample.noise)]\n    return (base.to(device), torch.from_numpy(sample.xyz).to(device),\n            neighbors.to(device), torch.from_numpy(sample.colors[condition]).to(device),\n            torch.from_numpy(sample.clean).to(device))\n\n\ndef chamfer(prediction, target, knn):\n    a = knn(prediction[None], target[None], K=1).dists\n    b = knn(target[None], prediction[None], K=1).dists\n    return a.mean() + b.mean()\n\n\ndef fit(mode, samples, prepared, epochs, seed, device, knn, directory):\n    torch.manual_seed(seed)\n    model = ResidualHead(mode).to(device)\n    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)\n    rng = np.random.default_rng(seed)\n    color_conditions = ("original", "jitter")\n    best_score, best_state = math.inf, None\n    history = []\n    for epoch in range(epochs):\n        started = time.monotonic()\n        model.train()\n        train_losses = []\n        for index in rng.permutation(len(samples["train"])):\n            sample = samples["train"][index]\n            condition = color_conditions[(epoch + index) % len(color_conditions)]\n            base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n            optim.zero_grad(set_to_none=True)\n            predicted, _ = model(base, xyz, neighbors, rgb)\n            loss = chamfer(predicted, target, knn)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f"Non-finite training loss for {mode} on {sample.oid}")\n            loss.backward()\n            optim.step()\n            train_losses.append(loss.item())\n        model.eval()\n        val_scores = []\n        with torch.no_grad():\n            for sample in samples["val"]:\n                for condition in color_conditions:\n                    base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n                    predicted, _ = model(base, xyz, neighbors, rgb)\n                    val_scores.append(chamfer(predicted, target, knn).item() * sample.metric_scale**2)\n        score = float(np.mean(val_scores))\n        if not math.isfinite(score):\n            raise RuntimeError(f"Non-finite validation score for {mode}")\n        history.append(dict(epoch=epoch + 1, train_cd=float(np.mean(train_losses)), validation_cd=score))\n        if score < best_score:\n            best_score = score\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n        seconds = time.monotonic() - started\n        print(f"{mode} · epoch {epoch + 1}/{epochs} · validation CD {score:.6g} · {seconds:.1f} s", flush=True)\n    torch.save(best_state, directory / f"{mode}.pt")\n    model.load_state_dict(best_state)\n    return model.eval(), history\n\n\n@torch.no_grad()\ndef evaluate(samples, heads, prepared, device, knn):\n    rows = []\n    for sample in samples:\n        for condition in ("original", "jitter"):\n            base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n            predictions = {"noisy": (xyz, None), "frozen_pgd": (base, None)}\n            for mode, head in heads.items():\n                predictions[mode] = head(base, xyz, neighbors, rgb)\n            for method, (predicted, gate) in predictions.items():\n                score = chamfer(predicted, target, knn).item() * sample.metric_scale**2\n                if not math.isfinite(score):\n                    raise RuntimeError(f"Non-finite test score for {method} on {sample.oid}")\n                rows.append(dict(object_id=sample.oid, group=sample.group, xyz_noise=sample.noise,\n                                 rgb=condition, method=method, cd_squared=score,\n                                 gate_mean=gate.mean().item() if method == "gated_rgb" else ""))\n    return rows\n\n\ndef write_csv(path, rows):\n    with path.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef self_test():\n    torch.manual_seed(42)\n    xyz = torch.randn(32, 3)\n    base = xyz * 0.99\n    rgb = torch.rand_like(xyz)\n    distances = torch.cdist(xyz, xyz)\n    distances.fill_diagonal_(float("inf"))\n    indices = distances.argsort(dim=1)[:, :NEIGHBORS]\n    counts = []\n    for mode in MODES:\n        head = ResidualHead(mode)\n        counts.append(sum(p.numel() for p in head.parameters()))\n        prediction, _ = head(base, xyz, indices, rgb)\n        assert torch.equal(prediction, base)\n        optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3)\n        target = xyz * 0.97 + rgb * 0.05\n        for _ in range(4):\n            optimizer.zero_grad()\n            prediction, _ = head(base, xyz, indices, rgb)\n            (prediction - target).square().mean().backward()\n            assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in head.parameters())\n            optimizer.step()\n        prediction, gate = head(base, xyz, indices, rgb)\n        other, _ = head(base, xyz, indices, 1 - rgb)\n        assert prediction.shape == xyz.shape and torch.isfinite(prediction).all()\n        assert (0 < gate).all() and (gate < 1).all()\n        if mode == "geometry":\n            assert torch.equal(prediction, other)\n        else:\n            assert not torch.equal(prediction, other)\n            constant_a, _ = head(base, xyz, indices, torch.ones_like(rgb))\n            constant_b, _ = head(base, xyz, indices, torch.zeros_like(rgb))\n            assert torch.equal(constant_a, constant_b)\n        if mode == "gated_rgb":\n            assert gate.std() > 0\n        else:\n            assert torch.equal(gate[0].expand_as(gate), gate)\n    assert len(set(counts)) == 1\n    print(f"PASS · all three heads trained · {counts[0]} trainable parameters each")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--self-test", action="store_true")\n    parser.add_argument("--data", type=Path)\n    parser.add_argument("--pgd", type=Path, default=Path("/content/PGD"))\n    parser.add_argument("--out", type=Path)\n    parser.add_argument("--cache", type=Path)\n    parser.add_argument("--epochs", type=int, default=20)\n    parser.add_argument("--iterations", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    if args.self_test:\n        self_test()\n        return\n    if args.data is None or args.out is None or args.epochs < 1 or args.iterations < 1:\n        parser.error("A training run requires --data, --out and positive training settings")\n    if not torch.cuda.is_available():\n        parser.error("The published PGD operators require a CUDA runtime")\n    from pytorch3d.ops import knn_points\n    device = torch.device("cuda")\n    checkpoint = args.pgd / "pretrained" / "PGD.ckpt"\n    samples = load_samples(args.data, args.smoke)\n    args.out.mkdir(parents=True, exist_ok=False)\n    cache = args.cache or args.out / "cache"\n    prepared, config = precompute(samples, args.pgd.resolve(), checkpoint, args.iterations, cache,\n                                  args.seed, device, knn_points)\n    epochs = min(args.epochs, 2) if args.smoke else args.epochs\n    heads, history = {}, {}\n    for mode in MODES:\n        heads[mode], history[mode] = fit(\n            mode, samples, prepared, epochs, args.seed, device, knn_points, args.out,\n        )\n    rows = evaluate(samples["test"], heads, prepared, device, knn_points)\n    write_csv(args.out / "per_object.csv", rows)\n    groups = defaultdict(list)\n    for row in rows:\n        for group in ("all", row["group"]):\n            key = (group, row["xyz_noise"], row["rgb"], row["method"])\n            groups[key].append(row["cd_squared"])\n    write_csv(args.out / "summary.csv", [\n        dict(group=group, xyz_noise=noise, rgb=rgb, method=method, n_objects=len(values),\n             mean_cd_squared=float(np.mean(values)))\n        for (group, noise, rgb, method), values in sorted(groups.items())\n    ])\n    metadata = dict(pgd=config, epochs=epochs, seed=args.seed, smoke=args.smoke,\n                    selection_sha256=digest_file(args.data / "selection.json"), history=history,\n                    objects={split: sorted({s.oid for s in subset}) for split, subset in samples.items()})\n    (args.out / "run.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(f"COMPLETE · {\'smoke\' if args.smoke else \'full\'} · {len(rows)} test rows · {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'task4/evaluate_surface.py': '"""Measure the completed GSO experiment against its original triangle surfaces."""\nimport argparse\nfrom collections import defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport tarfile\n\nimport numpy as np\nimport pandas as pd\nimport point_cloud_utils as pcu\nimport requests\nimport torch\nimport trimesh\nfrom scipy.spatial import cKDTree\n\nfrom train_rgb_pgd import MODES, XYZ_LEVELS, NEIGHBORS, ResidualHead, digest_file, load_cloud\n\n\ndef find_shards(ids, candidates):\n    mapping = defaultdict(set)\n    found = set()\n    for archive in sorted(candidates.glob("*.npz")):\n        with np.load(archive, allow_pickle=False) as data:\n            matching = ids.intersection(data["ids"].tolist())\n        if found.intersection(matching):\n            raise ValueError("Duplicate test IDs in the candidate archives")\n        if matching:\n            mapping[int(archive.stem)].update(matching)\n            found.update(matching)\n        if found == ids:\n            return mapping\n    raise FileNotFoundError(f"Missing source shard for {sorted(ids - found)}")\n\n\ndef original_meshes(source, revision, mapping, directory):\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def retrieve(item):\n        index, objects = item\n        pending = {oid for oid in objects if not (directory / f"{oid}.glb").is_file()}\n        if not pending:\n            return\n        archive = f"gso-train-{index:05d}.tar"\n        url = f"https://huggingface.co/datasets/{source}/resolve/{revision}/data/{archive}"\n        for attempt in range(3):\n            by_stem = defaultdict(dict)\n            try:\n                with requests.get(url, stream=True, timeout=(20, 90)) as response:\n                    response.raise_for_status()\n                    with tarfile.open(fileobj=response.raw, mode="r|*") as tar:\n                        for member in tar:\n                            stem, _, suffix = Path(member.name).name.partition(".")\n                            if not member.isfile() or suffix not in {"glb", "json"}:\n                                continue\n                            if stem not in pending:\n                                continue\n                            content = tar.extractfile(member).read()\n                            entry = by_stem[stem]\n                            entry[suffix] = content\n                            if len(entry) == 2:\n                                oid = str(json.loads(entry["json"])["object_id"])\n                                if oid != stem:\n                                    raise ValueError(f"{archive}: metadata and archive key differ for {stem}")\n                                path = directory / f"{oid}.glb"\n                                partial = path.with_suffix(".part")\n                                partial.write_bytes(entry["glb"])\n                                partial.replace(path)\n                                pending.remove(oid)\n                                del by_stem[stem]\n                                if not pending:\n                                    break\n                if pending:\n                    raise ValueError(f"{archive}: did not find meshes for {sorted(pending)}")\n                print(f"{archive} · {len(objects)} reference mesh(es) ready", flush=True)\n                return\n            except (requests.RequestException, OSError, tarfile.ReadError):\n                if attempt == 2:\n                    raise\n\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        list(pool.map(retrieve, sorted(mapping.items())))\n\n\ndef normalize(xyz):\n    center = (xyz.max(0) + xyz.min(0)) / 2\n    radius = np.linalg.norm(xyz - center, axis=1).max()\n    if radius <= 0 or not np.isfinite(radius):\n        raise ValueError("Degenerate cloud")\n    return center, radius\n\n\ndef point_to_surface(points, vertices, faces):\n    _, fid, bc = pcu.closest_points_on_mesh(np.ascontiguousarray(points), vertices, faces)\n    if np.any(fid < 0) or not np.isfinite(bc).all():\n        raise ValueError("Failed closest-point query")\n    closest = pcu.interpolate_barycentric_coords(faces, fid, bc, vertices)\n    return np.sum((points - closest)**2, axis=-1)\n\n\ndef measure(args):\n    meta = json.loads((args.run / "run.json").read_text())\n    if meta["smoke"] or digest_file(args.data / "selection.json") != meta["selection_sha256"]:\n        raise ValueError("This full run does not match the supplied Task 3 split")\n    manifest = json.loads((args.data / "manifest.json").read_text())\n    selections = json.loads((args.data / "selection.json").read_text())\n    ids = set(meta["objects"]["test"])\n    if ids != {oid for oid, info in selections.items() if info["split"] == "test"} or len(ids) != 12:\n        raise ValueError("Run and dataset disagree about held-out objects")\n    if any(Path(oid).name != oid or oid in {".", ".."} for oid in ids):\n        raise ValueError("Invalid object ID")\n    reference = pd.read_csv(args.run / "per_object.csv").set_index(\n        ["object_id", "xyz_noise", "rgb", "method"], verify_integrity=True\n    )\n    if len(reference) != 360:\n        raise ValueError("Full test CSV must have 360 rows")\n    masks = {}\n    if args.point_masks is not None:\n        with np.load(args.point_masks, allow_pickle=False) as mask_archive:\n            masks = {oid: mask_archive[oid] for oid in mask_archive.files}\n        if not masks or not set(masks) <= ids:\n            raise ValueError("Point masks must name held-out objects")\n        ids = set(masks)\n    mesh_dir = args.mesh_cache / manifest["revision"]\n    original_meshes(manifest["source"], manifest["revision"], find_shards(ids, args.candidates), mesh_dir)\n\n    torch.set_num_threads(2)\n    heads = {}\n    for mode in MODES:\n        head = ResidualHead(mode)\n        head.load_state_dict(torch.load(args.run / f"{mode}.pt", map_location="cpu", weights_only=True))\n        heads[mode] = head.eval()\n    cache_prefix = json.dumps(meta["pgd"], sort_keys=True).encode()\n    rows, mesh_checks = [], []\n    with torch.no_grad():\n        for oid in sorted(ids):\n            folder = args.data / "test" / oid\n            clean, _ = load_cloud(folder / "clean.npz")\n            clean_center, clean_radius = normalize(clean)\n            scene = trimesh.load(io.BytesIO((mesh_dir / f"{oid}.glb").read_bytes()), file_type="glb", force="scene")\n            mesh = trimesh.util.concatenate([m for m in scene.dump() if isinstance(m, trimesh.Trimesh)])\n            vertices = np.ascontiguousarray((mesh.vertices - clean_center) / clean_radius, dtype=np.float64)\n            faces = np.ascontiguousarray(mesh.faces, dtype=np.int32)\n            points_per_cloud = len(clean)\n            mask = masks.get(oid)\n            if mask is not None:\n                if mask.dtype != np.bool_ or mask.shape != (len(clean),) or not mask.any():\n                    raise ValueError(f"Invalid clean-point mask for {oid}")\n                points_per_cloud = int(mask.sum())\n            all_points = [(clean.astype(np.float64) - clean_center) / clean_radius]\n            object_rows = []\n            for noise in XYZ_LEVELS:\n                noisy, original = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                same_noisy, jitter = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n                if not np.array_equal(noisy, same_noisy):\n                    raise ValueError(f"XYZ differs between RGB settings for {oid}")\n                center, radius = normalize(noisy)\n                xyz = (noisy - center) / radius\n                key = hashlib.sha256(cache_prefix + xyz.tobytes()).hexdigest()\n                pgd = np.load(args.pgd_cache / f"{key}.npy", allow_pickle=False)\n                x = torch.from_numpy(xyz)\n                base = torch.from_numpy(pgd)\n                target = (clean - center) / radius\n                target_tree = cKDTree(target)\n                _, neighbor_ids = cKDTree(xyz).query(xyz, k=NEIGHBORS + 1)\n                neighbors = torch.from_numpy(np.asarray(\n                    [row[row != i][:NEIGHBORS] for i, row in enumerate(neighbor_ids)]\n                ))\n                for condition, rgb in [("original", original), ("jitter", jitter)]:\n                    colors = torch.from_numpy(rgb)\n                    predictions = {"noisy": x, "frozen_pgd": base}\n                    predictions.update({mode: head(base, x, neighbors, colors)[0] for mode, head in heads.items()})\n                    for method, prediction in predictions.items():\n                        saved = reference.loc[(oid, noise, condition, method)]\n                        predicted = prediction.numpy()\n                        a, _ = target_tree.query(predicted)\n                        b, _ = cKDTree(predicted).query(target)\n                        score = (np.mean(a**2) + np.mean(b**2)) * float(radius / clean_radius)**2\n                        if not np.isclose(score, saved.cd_squared, rtol=1e-4, atol=1e-8):\n                            raise ValueError(\n                                f"Recomputed Chamfer mismatch for {oid}/{noise}/{condition}/{method}: "\n                                f"{score:g} vs {saved.cd_squared:g}"\n                            )\n                        row = dict(object_id=oid, group=saved.group, xyz_noise=noise, rgb=condition, method=method)\n                        if mask is None:\n                            row["cd_squared"] = saved.cd_squared\n                        else:\n                            row["n_points"] = points_per_cloud\n                        normalized = (predicted.astype(np.float64) * radius + center - clean_center) / clean_radius\n                        all_points.append(normalized)\n                        object_rows.append(row)\n            if mask is not None:\n                all_points = [points[mask] for points in all_points]\n            squared = point_to_surface(np.concatenate(all_points), vertices, faces)\n            squared = squared.reshape(len(all_points), points_per_cloud)\n            aligned = float(np.sqrt(squared[0].mean()))\n            if aligned > 1e-5 or not np.isfinite(squared).all():\n                raise ValueError(f"Reference mesh misalignment for {oid}: clean RMS = {aligned:g}")\n            mesh_checks.append(dict(object_id=oid, n_faces=len(faces), clean_rms=aligned))\n            for row, distances in zip(object_rows, squared[1:]):\n                row["p2surface_mean"] = float(np.sqrt(distances).mean())\n                row["p2surface_squared"] = float(distances.mean())\n            rows.extend(object_rows)\n            print(f"{len(mesh_checks)}/{len(ids)} · {oid} · clean reference RMS {aligned:.3g}", flush=True)\n    result = pd.DataFrame(rows)\n    expected = reference.loc[reference.index.get_level_values("object_id").isin(ids)]\n    actual = result.set_index(reference.index.names, verify_integrity=True)\n    if not actual.index.sort_values().equals(expected.index.sort_values()):\n        raise ValueError("Surface rows do not cover the selected test cases")\n    metrics = ["p2surface_mean", "p2surface_squared"]\n    if not masks:\n        metrics.insert(0, "cd_squared")\n    args.out.mkdir(parents=True, exist_ok=True)\n    result.to_csv(args.out / "per_object_surface.csv", index=False)\n    pd.DataFrame(mesh_checks).to_csv(args.out / "reference_mesh_checks.csv", index=False)\n    result.groupby(["xyz_noise", "rgb", "method"])[metrics].mean().to_csv(args.out / "surface_by_condition.csv")\n    result.groupby(["group", "method"])[metrics].mean().to_csv(args.out / "surface_by_group.csv")\n    print("Overall means:\\n" + result.groupby("method")[metrics].mean().to_string(float_format="%.9g"))\n    print("Group means, squared P2S × 1e6:\\n" +\n          (result.groupby(["group", "method"]).p2surface_squared.mean().unstack()*1e6).to_string(float_format="%.6g"))\n    print(f"COMPLETE · {len(result)} surface scores · {args.out}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "run", "pgd-cache", "candidates", "mesh-cache", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    parser.add_argument("--point-masks", type=Path, help="NPZ mapping held-out object IDs to clean-point masks")\n    measure(parser.parse_args())\n', 'task4/texture_masks.py': '"""Recreate the original GSO texture-only masks from the Task 3 candidates."""\nimport argparse\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nfrom threadpoolctl import threadpool_limits\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "data"))\nfrom ranking import compute_scores, compute_thresholds\n\nCANDIDATE_COUNT = 1030\nSHARD_COUNT = 43\nNEIGHBOURS = 24\n\n\ndef generate(data, candidates, out):\n    manifest = json.loads((data / "manifest.json").read_text())\n    selection = json.loads((data / "selection.json").read_text())\n    targets = {\n        oid for oid, item in selection.items()\n        if item["split"] == "test" and item["group"] == "texture_only"\n    }\n    if len(targets) != 2 or manifest["skipped"]:\n        raise ValueError("Expected the original complete GSO run and its two texture-only test shapes")\n    shards = sorted(candidates.glob("*.npz"))\n    if len(shards) != SHARD_COUNT:\n        raise ValueError(f"Expected {SHARD_COUNT} Task 3 candidate shards in {candidates}")\n\n    scores = {}\n    with threadpool_limits(limits=2):\n        for index, shard in enumerate(shards, 1):\n            with np.load(shard, allow_pickle=False) as archive:\n                if len(archive["skipped"]):\n                    raise ValueError(f"Missing candidate objects in {shard}")\n                for name, xyz, rgb in zip(archive["ids"], archive["xyz"], archive["rgb"], strict=True):\n                    oid = str(name)\n                    if oid in scores:\n                        raise ValueError(f"Duplicate candidate object ID {oid}")\n                    g, c = compute_scores(xyz, rgb, k=NEIGHBOURS)\n                    if not (np.isfinite(g).all() and np.isfinite(c).all()):\n                        raise ValueError(f"Invalid candidate scores for {oid}")\n                    scores[oid] = (g, c)\n                    if oid in targets:\n                        with np.load(data / "test" / oid / "clean.npz", allow_pickle=False) as clean:\n                            if not (np.array_equal(clean["xyz"], xyz) and np.array_equal(clean["rgb"], rgb)):\n                                raise ValueError(f"Candidate and clean test cloud differ for {oid}")\n            if index % 7 == 0 or index == len(shards):\n                print(f"{index}/{len(shards)} candidate archives · {len(scores)} objects", flush=True)\n\n    if len(scores) != CANDIDATE_COUNT or not targets <= scores.keys():\n        raise ValueError("Incomplete Task 3 candidate cache")\n    thresholds = compute_thresholds(scores)\n    geometry_p50 = float(thresholds["g_low"])\n    colour_p90 = float(thresholds["c_high"])\n    masks = {oid: (scores[oid][0] <= geometry_p50) & (scores[oid][1] >= colour_p90)\n             for oid in sorted(targets)}\n    if not all(mask.any() for mask in masks.values()):\n        raise ValueError("Texture-only mask is empty")\n\n    metadata = dict(revision=manifest["revision"], candidate_objects=len(scores), neighbours=NEIGHBOURS,\n                    geometry_p50=geometry_p50, colour_p90=colour_p90,\n                    selected_points={oid: int(mask.sum()) for oid, mask in masks.items()},\n                    numpy=version("numpy"), scikit_learn=version("scikit-learn"))\n    out.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(out / "clean_point_masks.npz", **masks)\n    (out / "mask_metadata.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(json.dumps(metadata, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for flag in ("data", "candidates", "out"):\n        parser.add_argument(f"--{flag}", type=Path, required=True)\n    args = parser.parse_args()\n    generate(args.data, args.candidates, args.out)\n'}
work = Path('/content/mvp_surface_evaluation')
for name, source in sources.items():
    path = work / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)

env = dict(os.environ, OPENBLAS_NUM_THREADS='2', OMP_NUM_THREADS='2',
           MKL_NUM_THREADS='2')
mesh_cache = work / 'meshes'

def execute(script, **flags):
    argv = [sys.executable, '-u', str(work / script)]
    for flag, value in flags.items():
        argv.extend(['--' + flag.replace('_', '-'), str(value)])
    with subprocess.Popen(argv, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
        if proc.wait():
            raise subprocess.CalledProcessError(proc.returncode, argv)


In [ ]:
surface = run / 'surface'
texture = surface / 'texture_regions'
if not (surface / 'per_object_surface.csv').is_file():
    execute('task4/evaluate_surface.py', data=data, run=run, pgd_cache=pgd_cache,
            candidates=candidates, mesh_cache=mesh_cache, out=surface)
if not all((texture / name).is_file() for name in ('clean_point_masks.npz', 'mask_metadata.json')):
    execute('task4/texture_masks.py', data=data, candidates=candidates, out=texture)


In [ ]:
execute('task4/evaluate_surface.py', data=data, run=run, pgd_cache=pgd_cache,
        candidates=candidates, mesh_cache=mesh_cache, out=texture,
        point_masks=texture / 'clean_point_masks.npz')

import pandas as pd
from IPython.display import display
for folder in (surface, texture):
    rows = pd.read_csv(folder / 'per_object_surface.csv')
    columns = [c for c in ('cd_squared', 'p2surface_mean', 'p2surface_squared') if c in rows]
    print(folder)
    display(rows.groupby('method')[columns].mean())
